In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.linalg import inv, matrix_rank, det

print('NumPy  :', np.__version__)
print('Pandas :', pd.__version__)

NumPy  : 2.4.2
Pandas : 2.3.3


In [34]:
import numpy as np
from numpy.linalg import inv, matrix_rank

def det_checker(X):
    m, d = X.shape
    if m == d:   return "even"
    elif m > d:  return "over"
    else:        return "under"

def RC_checker(X, y):
    X_aug  = np.append(X, y, axis=1)
    rankX  = matrix_rank(X)
    rankX_ = matrix_rank(X_aug)
    d = X.shape[1]
    if rankX == rankX_:
        RC = 1 if rankX == d else 3
    else:
        RC = 2
    return RC, rankX, rankX_

def evenSolver(X, y):
    RC, _, _ = RC_checker(X, y)
    if RC == 1:   return inv(X) @ y, "Unique solution."
    elif RC == 2: return None, "No solution."
    else:         return None, "Infinitely many solutions."

def overSolver(X, y):
    RC, _, _ = RC_checker(X, y)
    if RC == 1:   return inv(X.T @ X) @ X.T @ y, "Unique solution."
    elif RC == 3: return None, "Infinitely many solutions."
    else:         return inv(X.T @ X) @ X.T @ y, "No exact sol, least square approx."

def underSolver(X, y):
    RC, _, _ = RC_checker(X, y)
    if RC == 2:   return None, "No solution."
    else:         return X.T @ inv(X @ X.T) @ y, "No exact sol, least norm approx."

def solveLE(X, y):
    det = det_checker(X)
    if det == "even":   w, ans = evenSolver(X, y)
    elif det == "over": w, ans = overSolver(X, y)
    else:               w, ans = underSolver(X, y)
    print(ans, "\nw =", w)
    return w

# --- Quick diagnostic ---
X = np.array([[1, 3], [1, 4], [1, 5], [1, 6], [1, 7]])
y = np.array([[5], [4], [3], [2], [1]])

print("System type:", det_checker(X))
RC, rX, rX_ = RC_checker(X, y)
print(f"rank(X)={rX}, rank(X~)={rX_}, case={RC}")
w = solveLE(X, y)

System type: over
rank(X)=2, rank(X~)=2, case=1
Unique solution. 
w = [[ 8.]
 [-1.]]


In [35]:
from sklearn.preprocessing import PolynomialFeatures
from numpy.linalg import inv
import numpy as np

def polyTx(X, order):
    """Polynomial feature matrix with bias column. Shape: (N, order+1)."""
    return PolynomialFeatures(order).fit_transform(X)

def solvePR(P, y, ridge=False, lamb=0.01):
    """Solve polynomial regression. Auto primal (N>M) or dual (N<M)."""
    if ridge:
        if P.shape[0] > P.shape[1]:   # Primal
            w = inv(P.T @ P + lamb * np.eye(P.shape[1])) @ P.T @ y
        else:                          # Dual
            w = P.T @ inv(P @ P.T + lamb * np.eye(P.shape[0])) @ y
    else:
        if P.shape[0] > P.shape[1]:   # Primal
            w = inv(P.T @ P) @ P.T @ y
        else:                          # Dual
            w = P.T @ inv(P @ P.T) @ y
    return w

def solveLE_Ridge(X, y, lamb=0.01):
    """Linear regression with Ridge. X must already include bias column."""
    return solvePR(X, y, ridge=True, lamb=lamb)

# Quick test
X_test_poly = np.array([[-10], [-8], [-3], [-1], [2], [8]])
y_test_poly = np.array([[5], [5], [4], [3], [2], [2]])
P = polyTx(X_test_poly, 3)
print('P shape:', P.shape)  # (6, 4): [1, x, x^2, x^3]
w = solvePR(P, y_test_poly)
print('w:', w.ravel())

P shape: (6, 4)
w: [ 2.68935636 -0.37722517  0.01343815  0.00285772]


In [36]:
import numpy as np
import matplotlib.pyplot as plt

X = np.array([[1], [2], [2.9]])
y = np.array([[2], [4.1], [6.1]])

# --- With bias ---
bias = np.ones((X.shape[0], 1))
X_b  = np.hstack((bias, X))         # [1, x]
w    = solveLE(X_b, y)              # w = [w0_bias, w1_slope]
w

# # Predict for new inputs
# X_test   = np.array([[30], [5]])
# X_test_b = np.hstack((np.ones((X_test.shape[0], 1)), X_test))
# y_pred   = X_test_b @ w
# print('Predictions:', y_pred.ravel())

# # --- Without bias ---
# w_nb = solveLE(X, y)

# # --- Plot both ---
# plt.scatter(X, y, color='blue', label='Training Samples', marker='o')
# plt.plot(X, X_b @ w,    color='red',   label='With bias')
# plt.plot(X, X   @ w_nb, color='green', label='No bias')
# plt.xlabel('Students'); plt.ylabel('Books sold'); plt.legend(); plt.show()

No exact sol, least square approx. 
w = [[-0.17509225]
 [ 2.15682657]]


array([[-0.17509225],
       [ 2.15682657]])

In [37]:
import numpy as np
from numpy.linalg import inv

X = np.array([[1], [2], [2.9]])
y = np.array([[2], [4.1], [6.1]])

bias = np.ones((X.shape[0], 1))
X_b = np.hstack((bias, X))  # [1, x]


# --- Via tool (auto primal/dual) ---
lamb = 0.1  # 'lambda' is a Python keyword
w_ridge = solvePR(X_b, y, ridge=True, lamb=lamb)
print("Ridge w (1dp):", np.around(w_ridge.T, decimals=4))

# --- Manually (primal, N > M) ---
reg_L = lamb * np.eye(X_b.shape[1])  # lambda * I
w_primal = inv(X_b.T @ X_b + reg_L) @ X_b.T @ y
print("Manual primal w:", np.around(w_primal.T, decimals=4))

# # --- Manually (dual, N < M) ---
# X_under = np.array([[1, 0, 1], [1, -1, 1]])
# y_under = np.array([[0], [1]])
# P_under = polyTx(X_under, 3)
# reg_L_d = lamb * np.eye(P_under.shape[0])
# w_dual = P_under.T @ inv(P_under @ P_under.T + reg_L_d) @ y_under
# print("Dual w (1dp):", np.around(w_dual.T, decimals=1))

Ridge w (1dp): [[0.0383 2.0477]]
Manual primal w: [[0.0383 2.0477]]


In [38]:
# Predict for new inputs
X_test   = np.array([[1.5]])
X_test_b = np.hstack((np.ones((X_test.shape[0], 1)), X_test))
y_pred   = X_test_b @ w_ridge
print('Predictions:', y_pred.ravel())

Predictions: [3.10981474]


# Q2

In [ ]:
import numpy as np
from numpy.linalg import inv

X = np.array([[1, 2, 3], [4, 0, 6], [1, 1, 0], [0, 1, 2], [5, 7, -2], [-1, 4, 0]])
Y = np.array([[1, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1], [0, 1, 0], [0, 0, 1]])
## CAREFUL of bias
# --- With bias ---
bias = np.ones((X.shape[0], 1))
X_b = np.hstack((bias, X))  # [1, x]

W = solveLE(X_b, Y)  # weight matrix, shape (M, n_classes)

X_test = np.array([[1, 1, -2, 3]])
y_raw = X_test @ W
y_raw # Gives me 2


No exact sol, least square approx. 
w = [[-0.2424931   0.92743768  0.31505542]
 [ 0.01542144  0.18137978 -0.19680122]
 [ 0.09029091 -0.19338387  0.10309296]
 [ 0.21626451 -0.2752962   0.05903169]]


array([[0.24114004, 0.6696966 , 0.08916336]])

In [ ]:
order = 3
P = polyTx(X, order)  # shape (N, 4): [1, x, x^2, x^3]
print("NOTICE: PolynomialFeatures column order may differ from handwritten version!")
print("P:\n", P)

# Fit
w_poly = solvePR(P, Y)
print("Coefficients:", w_poly.ravel())

# Predict on new point
x_test = np.array([[1, -2, 3]])
P_test = polyTx(x_test, order)  # same order
y_pred_poly = P_test @ w_poly
y_pred_poly # Gives me 2 as well 

NOTICE: PolynomialFeatures column order may differ from handwritten version!
P:
 [[  1.   1.   2.   3.   1.   2.   3.   4.   6.   9.   1.   2.   3.   4.
    6.   9.   8.  12.  18.  27.]
 [  1.   4.   0.   6.  16.   0.  24.   0.   0.  36.  64.   0.  96.   0.
    0. 144.   0.   0.   0. 216.]
 [  1.   1.   1.   0.   1.   1.   0.   1.   0.   0.   1.   1.   0.   1.
    0.   0.   1.   0.   0.   0.]
 [  1.   0.   1.   2.   0.   0.   0.   1.   2.   4.   0.   0.   0.   0.
    0.   0.   1.   2.   4.   8.]
 [  1.   5.   7.  -2.  25.  35. -10.  49. -14.   4. 125. 175. -50. 245.
  -70.  20. 343. -98.  28.  -8.]
 [  1.  -1.   4.   0.   1.  -4.  -0.  16.   0.   0.  -1.   4.   0. -16.
   -0.  -0.  64.   0.   0.   0.]]
Coefficients: [-0.01176815  0.17482347  0.04418372  0.00193766  0.16820103 -0.0107976
 -0.00746191  0.17015972  0.03168837 -0.01416555 -0.00098195  0.06758816
 -0.00031705  0.15536443 -0.00719315  0.0069067   0.1410684  -0.02408252
  0.01156987  0.00117843 -0.04089203 -0.00069248  0.1431

array([[-0.51646251,  0.90008188,  2.00518001]])

## Q3

In [42]:
X = np.array([[1, 4], [5, -1], [2, 3]])
y = np.array([1, 3, 1]).reshape(3, 1)
X_aug = np.append(X, y, axis=1)
print("\nrank(X)  =", matrix_rank(X))
print("rank(X~) =", matrix_rank(X_aug))


rank(X)  = 2
rank(X~) = 3


In [44]:
order = 2
P = polyTx(X, order)
P_aug = np.append(P, y, axis=1)
print("\nrank(X)  =", matrix_rank(P))
print("rank(X~) =", matrix_rank(P_aug))


rank(X)  = 3
rank(X~) = 3


# Q9

In [ ]:
X = np.array([1, 3, 2, 3, 6, -4, 5, -1, 6, 9, 8, 0]).reshape(4, 3)
y = np.array([1, 3, 9, 3]).reshape(-1, 1)
X_aug = np.append(X, y, axis=1)
print("\nrank(X)  =", matrix_rank(X))
print("rank(X~) =", matrix_rank(X_aug))


rank(X)  = 3
rank(X~) = 4


# Q10

In [52]:
X = np.array([[2, 0, 0], [0, -1, 1]])
# det(X)